I'm following the tutorial on OC. https://openclassrooms.com/fr/courses/7410486-nettoyez-et-analysez-votre-jeu-de-donnees/7451506-nettoyez-vos-donnees-avec-python

Import the libraries

In [1]:
import pandas as pd
import numpy as np
import re

Load and look at the data

In [2]:
data = pd.read_csv('personnes.csv')
data.head(7)

,prenom,email,date_naissance,pays,taille
0,Leila,leila@example.com,23/01/1990,France,1.49m
1,Samuel,samuel_329@example.com,20/09/2001,NaN,1.67m
2,Radia,choupipoune@supermail.eu,12 sept. 1984,Côte d'ivoire,153cm
3,Marc,"marco23@example.com, mc23@supermail.eu",10/02/1978,France,1.65m
4,Heri,helloworld@supermail.eu,05/03/2008,Madagascar,1.34m
5,Hanna,hanna2019@supermail.eu,01/01/1970,24,3.45m
6,samuël,samuel_329@example.com,NaN,Bénin,1.45m


Let's see the info about this dataframe

In [3]:
data.describe()

,prenom,email,date_naissance,pays,taille
count,7,7,6,6,7
unique,7,6,6,5,7
top,Leila,samuel_329@example.com,23/01/1990,France,1.49m
freq,1,2,1,2,1


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   prenom          7 non-null      str  
 1   email           7 non-null      str  
 2   date_naissance  6 non-null      str  
 3   pays            6 non-null      str  
 4   taille          7 non-null      str  
dtypes: str(5)
memory usage: 412.0 bytes


## Check mistakes

1. Count empty values

In [5]:
data.isnull().sum()

prenom            0
email             0
date_naissance    1
pays              1
taille            0
dtype: int64

2. Check for duplicates

Simplest version

In [6]:
data[data['email'].duplicated(keep=False)]

,prenom,email,date_naissance,pays,taille
1,Samuel,samuel_329@example.com,20/09/2001,NaN,1.67m
6,samuël,samuel_329@example.com,NaN,Bénin,1.45m


Validating

In [7]:
data['email'].duplicated(keep=False)

0    False
1     True
2    False
3    False
4    False
5    False
6     True
Name: email, dtype: bool

In [ ]:
# Original

data.loc[data['email'].duplicated(keep=False),:]
#"keep=False" marks all duplicates True, 
# including the first one, 
# because by default it's False.
# loc filters rows (and columns) by label
# : means "keep all columns"

#Simpler
data.loc[data['email'].duplicated(keep=False)]
# ,: is redundant because .loc returns all columns
# when I pass only a row filter

## Fix the errors

### 1. Keep only the countries from the list.

In [8]:
data['pays']

0           France
1              NaN
2    Côte d'ivoire
3           France
4       Madagascar
5               24
6            Bénin
Name: pays, dtype: str

Cause we have an integer in one of the cells

Here's a shorter way to code it

In [9]:
VALID_COUNTRIES = ['France', 'Côte d\'ivoire', 'Madagascar', 'Bénin', 'Allemagne', 'USA']
data.loc[~data['pays'].isin(VALID_COUNTRIES), 'pays'] = np.nan

#As a result, we only keep these countries
data['pays']

0           France
1              NaN
2    Côte d'ivoire
3           France
4       Madagascar
5              NaN
6            Bénin
Name: pays, dtype: str

In [ ]:
#Original way
VALID_COUNTRIES = ['France', 'Côte d\'ivoire', 
                   'Madagascar', 'Bénin', 
                   'Allemagne', 'USA']
mask = ~data['pays'].isin(VALID_COUNTRIES)
data.loc[mask, 'pays'] = np.nan

### 2. E-mails

In [10]:
data['email']

0                         leila@example.com
1                    samuel_329@example.com
2                  choupipoune@supermail.eu
3    marco23@example.com, mc23@supermail.eu
4                   helloworld@supermail.eu
5                    hanna2019@supermail.eu
6                    samuel_329@example.com
Name: email, dtype: str

Keep only the first e-mail, if there's more than one

Simplest version

In [11]:
data['email'] = data['email'].str.partition(',')[0]
#.str.partition() splits on the first occurrence

#check the result
data['email']

0           leila@example.com
1      samuel_329@example.com
2    choupipoune@supermail.eu
3         marco23@example.com
4     helloworld@supermail.eu
5      hanna2019@supermail.eu
6      samuel_329@example.com
Name: email, dtype: str

In [ ]:
# original
data['email'] = data['email'].str.split(',', n=1, expand=True)[0]
# split(..., expand=True) - creates a full DataFrame
# then it grabs column [0]

#Simpler
data['email'] = data['email'].str.split(',').str[0]
# .str[0] picks the first element from the list
# no need to create a DataFrame

### 3. Fix the heights

In [12]:
data['taille']

0    1.49m
1    1.67m
2    153cm
3    1.65m
4    1.34m
5    3.45m
6    1.45m
Name: taille, dtype: str

#### 3.1 Convert heights to numbers

#### 3.2 Replace nulls with the average value

Simpler

In [13]:
# string without last character, convert to a number
data['taille'] = pd.to_numeric(data['taille'].str[:-1], errors = 'coerce')
#replace nulls with an average, round to two decimals
data['taille'] = data['taille'].fillna(data['taille'].mean()).round(2)

#check the results
data['taille']

0    1.49
1    1.67
2    1.84
3    1.65
4    1.34
5    3.45
6    1.45
Name: taille, dtype: float64

In [ ]:
#1st version
#Delete the last character (letter)
data['taille'] = data['taille'].str[:-1]

#Convert the column to numbers,
# Nulls for errors
data['taille'] = pd.to_numeric(data['taille'], errors = 'coerce')

#Replace nulls with the average value
data.loc[data['taille'].isnull(), 'taille'] = data['taille'].mean()

### 4. Fix the dates

In [14]:
data['date_naissance']

0       23/01/1990
1       20/09/2001
2    12 sept. 1984
3       10/02/1978
4       05/03/2008
5       01/01/1970
6              NaN
Name: date_naissance, dtype: str

In [15]:
data['date_naissance'] = pd.to_datetime(
    data['date_naissance'], format='%d/%m/%Y',
         errors = 'coerce')

#validate
data['date_naissance']

0   1990-01-23
1   2001-09-20
2          NaT
3   1978-02-10
4   2008-03-05
5   1970-01-01
6          NaT
Name: date_naissance, dtype: datetime64[us]

## Let's see the changes

In [16]:
data.head(7)

,prenom,email,date_naissance,pays,taille
0,Leila,leila@example.com,1990-01-23,France,1.49
1,Samuel,samuel_329@example.com,2001-09-20,NaN,1.67
2,Radia,choupipoune@supermail.eu,NaT,Côte d'ivoire,1.84
3,Marc,marco23@example.com,1978-02-10,France,1.65
4,Heri,helloworld@supermail.eu,2008-03-05,Madagascar,1.34
5,Hanna,hanna2019@supermail.eu,1970-01-01,NaN,3.45
6,samuël,samuel_329@example.com,NaT,Bénin,1.45


In [17]:
data.describe()

,date_naissance,taille
count,5,7.000000
mean,1989-08-12 09:36:00,1.841429
min,1970-01-01 00:00:00,1.340000
25%,1978-02-10 00:00:00,1.470000
50%,1990-01-23 00:00:00,1.650000
75%,2001-09-20 00:00:00,1.755000
max,2008-03-05 00:00:00,3.450000
std,NaN,0.728204


In [18]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   prenom          7 non-null      str           
 1   email           7 non-null      str           
 2   date_naissance  5 non-null      datetime64[us]
 3   pays            5 non-null      str           
 4   taille          7 non-null      float64       
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 412.0 bytes
